# Train Classifier on BirdNET Embeddings

This notebook trains a shallow classifier on pre-computed BirdNET embeddings using the train/test/validation splits from YOLO.

In [58]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, multilabel_confusion_matrix
from sklearn.multioutput import MultiOutputClassifier
import matplotlib.pyplot as plt
import seaborn as sns

## Load BirdNET Embeddings

In [59]:
# Load BirdNET embeddings (one embedding per individual call clip)
embeddings_path = '/home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/All_call_classes/Split_data_controlled/OpenSoundScape/birds_embeddings_birdnet_padded_3s.csv'
embeddings_df = pd.read_csv(embeddings_path)

# Extract trial ID from the file path to match with YOLO splits
# Example: .../SL07_Trial_1_trim_37.44715902_37.57169857_D.WAV -> SL07_Trial_1_trim
def extract_trial_id(filepath):
    filename = filepath.split('/')[-1]  # Get filename
    # Remove the timestamp and call type parts: file_id_begin_end_calltype.WAV
    parts = filename.rsplit('_', 3)  # Split from right to remove begin_end_calltype
    return parts[0]  # Return just the trial ID

embeddings_df['trial_id'] = embeddings_df['file'].apply(extract_trial_id)

# Use the calltype column that already exists in the CSV
print(f"Loaded embeddings: {embeddings_df.shape}")
print(f"Call types found: {embeddings_df['calltype'].unique()}")
print(f"Sample trial IDs: {embeddings_df['trial_id'].unique()[:5]}")
embeddings_df.head()

Loaded embeddings: (6047, 1029)
Call types found: <StringArray>
[    'D',     'F', 'Cheer', 'Check',     'I', 'Chits',     'E',     'G',
     'N',     'J',     'B',     'K',     'C',     'L',     'A',     'M',
     'O',     'H', 'Growl']
Length: 19, dtype: str
Sample trial IDs: <StringArray>
[ 'SL07_Trial_1_trim', 'SL17a_Trial_1_trim',  'SL10_Trial_1_trim',
  'AZ13_Trial_3_trim',  'SL51_Trial_1_trim']
Length: 5, dtype: str


/tmp/ipykernel_552606/667256170.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  embeddings_df['trial_id'] = embeddings_df['file'].apply(extract_trial_id)


,calltype,file,start_time,end_time,0,1,2,3,4,5,...,1015,1016,1017,1018,1019,1020,1021,1022,1023,trial_id
0,D,/home/Shelby/blackbird_calls/Experiments/Detec...,0.0,3.0,0.067894,0.344097,0.135184,0.049056,0.659216,1.029421,...,0.000000,0.283043,0.353124,0.850524,0.342322,0.000000,1.061636,0.762078,0.485685,SL07_Trial_1_trim
1,F,/home/Shelby/blackbird_calls/Experiments/Detec...,0.0,3.0,0.027922,0.142469,0.005187,0.067271,0.845247,0.754935,...,0.000000,0.565755,0.272597,0.313492,0.231603,0.067483,1.466044,1.240883,0.785682,SL17a_Trial_1_trim
2,F,/home/Shelby/blackbird_calls/Experiments/Detec...,0.0,3.0,0.063137,0.392312,0.081558,0.013896,0.519530,0.799999,...,0.083630,0.350261,0.338610,0.525507,0.445224,0.000000,0.847136,0.351975,0.716970,SL10_Trial_1_trim
3,Cheer,/home/Shelby/blackbird_calls/Experiments/Detec...,0.0,3.0,0.247558,0.025925,0.128917,0.579505,0.000000,0.203117,...,0.000000,0.057046,0.103796,1.025722,0.991909,0.000000,0.515161,0.025441,1.636110,SL10_Trial_1_trim
4,Check,/home/Shelby/blackbird_calls/Experiments/Detec...,0.0,3.0,0.023261,0.383073,0.003367,0.000000,0.256864,0.177266,...,0.000597,1.142428,0.817335,0.477512,0.280269,0.000000,2.258079,0.730854,0.299776,AZ13_Trial_3_trim


## Load Train/Test/Validation Splits

In [60]:
# Load YOLO split files to get trial IDs for train/test/val
import os

split_dir = '/home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/All_call_classes/Split_data_controlled/YOLO/dataset_split/'

def extract_trial_ids_from_yolo(txt_file):
    """Extract unique trial identifiers from YOLO split file"""
    with open(txt_file, 'r') as f:
        paths = [line.strip() for line in f]
    
    trial_ids = set()
    for path in paths:
        filename = os.path.basename(path)
        # Remove _spec_XX-XX.png to get trial ID
        trial_id = filename.rsplit('_spec_', 1)[0]
        trial_ids.add(trial_id)
    
    return trial_ids

# Get trial IDs for each split
test_trials = extract_trial_ids_from_yolo(os.path.join(split_dir, 'test.txt'))
val_trials = extract_trial_ids_from_yolo(os.path.join(split_dir, 'validate.txt'))
train_trials = extract_trial_ids_from_yolo(os.path.join(split_dir, 'train.txt'))

print(f"Test trials ({len(test_trials)}): {sorted(test_trials)[:5]}...")
print(f"Val trials ({len(val_trials)}): {sorted(val_trials)[:5]}...")
print(f"Train trials ({len(train_trials)}): {sorted(list(train_trials)[:5])}...")

Test trials (16): ['AZ02_Trial_1_trim', 'AZ16_Trial_1_trim', 'SL07_Trial_2_trim', 'SL23_Trial_1_trim', 'SL37_Trial_1_trim']...
Val trials (16): ['AZ08_Trial_2_trim', 'AZ15_Trial_2_trim', 'EF01_Trial_1_trim', 'EF04_Trial_1_trim', 'SL01_Trial_2_trim']...
Train trials (78): ['AZ13_Trial_2_trim', 'EF01_Trial_2_trim', 'EF01_Trial_3_trim', 'SL11_Trial_3_trim', 'SL17a_Trial_2_trim']...


## Split Data by Trial IDs

In [61]:
# Split embeddings by trial ID
train_mask = embeddings_df['trial_id'].isin(train_trials)
val_mask = embeddings_df['trial_id'].isin(val_trials)
test_mask = embeddings_df['trial_id'].isin(test_trials)

train_df = embeddings_df[train_mask]
val_df = embeddings_df[val_mask]
test_df = embeddings_df[test_mask]

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")
print(f"Total: {len(train_df) + len(val_df) + len(test_df)} / {len(embeddings_df)}")

# Prepare X (embeddings) and y (labels)
# Embedding columns are numbered 0-1023
embedding_cols = [str(i) for i in range(1024)]

X_train = train_df[embedding_cols].values
y_train = train_df['calltype'].values

X_val = val_df[embedding_cols].values
y_val = val_df['calltype'].values

X_test = test_df[embedding_cols].values
y_test = test_df['calltype'].values

print(f"\nX_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}, y_val shape: {y_val.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

Training samples: 3932
Validation samples: 1144
Test samples: 952
Total: 6028 / 6047

X_train shape: (3932, 1024), y_train shape: (3932,)
X_val shape: (1144, 1024), y_val shape: (1144,)
X_test shape: (952, 1024), y_test shape: (952,)


In [62]:
# Debug: Check the format of indices
print("Sample embeddings indices:")
print(embeddings_df.index[:5].tolist())
print("\nSample train_df indices:")
print(train_df.index[:5].tolist())
print("\nAre they the same type?")
print(f"Embeddings index type: {type(embeddings_df.index[0])}")
print(f"Labels index type: {type(train_df.index[0])}")

Sample embeddings indices:
[0, 1, 2, 3, 4]

Sample train_df indices:
[0, 1, 2, 3, 4]

Are they the same type?
Embeddings index type: <class 'int'>
Labels index type: <class 'numpy.int64'>


In [63]:
# Check the structure of the embeddings file more carefully
print("Embeddings DataFrame info:")
print(f"Shape: {embeddings_df.shape}")
print(f"\nFirst few column names:")
print(embeddings_df.columns[:10].tolist())
print(f"\nFirst few index values:")
print(embeddings_df.index[:10].tolist())
print(f"\nDoes this look like it has audio file paths as index? Or call types?")

Embeddings DataFrame info:
Shape: (6047, 1029)

First few column names:
['calltype', 'file', 'start_time', 'end_time', '0', '1', '2', '3', '4', '5']

First few index values:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

Does this look like it has audio file paths as index? Or call types?


## Check Call Type Distribution

In [64]:
# Check call type distribution in each split
from collections import Counter

print("Call type distribution in training set:")
train_counts = Counter(y_train)
for call_type, count in sorted(train_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {call_type}: {count}")

print("\nCall type distribution in validation set:")
val_counts = Counter(y_val)
for call_type, count in sorted(val_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {call_type}: {count}")

print("\nCall type distribution in test set:")
test_counts = Counter(y_test)
for call_type, count in sorted(test_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {call_type}: {count}")

Call type distribution in training set:
  Check: 2333
  Cheer: 484
  F: 242
  E: 178
  D: 169
  Chits: 125
  C: 93
  K: 88
  M: 55
  G: 47
  I: 26
  N: 20
  J: 19
  B: 17
  O: 14
  A: 8
  H: 6
  L: 4
  Growl: 4

Call type distribution in validation set:
  Check: 757
  C: 90
  B: 74
  I: 41
  Cheer: 37
  D: 30
  L: 22
  J: 19
  Chits: 19
  G: 18
  E: 12
  K: 9
  O: 6
  F: 4
  H: 4
  M: 1
  Growl: 1

Call type distribution in test set:
  Check: 540
  Cheer: 186
  E: 58
  B: 45
  K: 33
  C: 19
  Chits: 18
  G: 12
  A: 12
  F: 10
  I: 8
  O: 4
  H: 4
  D: 2
  J: 1


## Train Shallow classifier

In [ ]:
# create a neural network classifier to take in the embeddings and classify call types
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

def create_dataloader(X, y, batch_size=32, shuffle=True):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(pd.factorize(y)[0], dtype=torch.long)  # Convert labels to integers
    dataset = TensorDataset(X_tensor, y_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


class EmbeddingClassifier(nn.Module):
    def __init__(self, input_dim, num_classes, dropout_rate=0.3):
        super(EmbeddingClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, input_dim // 2)
        self.bn1 = nn.BatchNorm1d(input_dim // 2)
        self.relu = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout_rate)
        
        self.fc2 = nn.Linear(input_dim // 2, input_dim // 4)
        self.bn2 = nn.BatchNorm1d(input_dim // 4)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout_rate)
        
        self.fc3 = nn.Linear(input_dim // 4, input_dim // 8)
        self.bn3 = nn.BatchNorm1d(input_dim // 8)
        self.relu3 = nn.ReLU()
        self.dropout3 = nn.Dropout(dropout_rate)
        
        self.fc4 = nn.Linear(input_dim // 8, num_classes)
        # Don't use softmax here - CrossEntropyLoss expects raw logits!
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout1(x)
        
        x = self.fc2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        
        x = self.fc3(x)
        x = self.bn3(x)
        x = self.relu3(x)
        x = self.dropout3(x)
        
        x = self.fc4(x)
        return x  # Return raw logits, not softmax probabilities

### Hyperparameters

In [ ]:
# Hyperparameters
input_dim = X_train.shape[1]
num_classes = len(set(y_train))
batch_size = 64
learning_rate = 0.001
num_epochs = 350
gpu = torch.cuda.is_available()

# regularization techniques
dropout_rate = 0.3
weight_decay = 1e-4  # L2 regularization
early_stopping_patience = 25

### Dataloaders

In [67]:
# Create dataloaders
train_loader = create_dataloader(X_train, y_train, batch_size=batch_size, shuffle=True)
val_loader = create_dataloader(X_val, y_val, batch_size=batch_size, shuffle=True)
test_loader = create_dataloader(X_test, y_test, batch_size=batch_size, shuffle=False)

### Initialize model

In [ ]:
# Initialize model, loss function, optimizer
model = EmbeddingClassifier(input_dim, num_classes, dropout_rate=dropout_rate)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

### train

In [ ]:
# Training loop with early stopping
best_val_loss = float('inf')
patience_counter = 0
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    # Training phase
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    
    epoch_loss = running_loss / len(train_loader.dataset)
    train_losses.append(epoch_loss)
    
    # Validation phase (happens after each epoch)
    model.eval()
    val_running_loss = 0.0
    with torch.no_grad():
        for inputs, labels in val_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_running_loss += loss.item() * inputs.size(0)
    
    val_epoch_loss = val_running_loss / len(val_loader.dataset)
    val_losses.append(val_epoch_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {epoch_loss:.4f}, Val Loss: {val_epoch_loss:.4f}")
    
    # Early stopping
    if val_epoch_loss < best_val_loss:
        best_val_loss = val_epoch_loss
        patience_counter = 0
        # Save best model
        best_model_state = model.state_dict().copy()
        print(f"  -> New best validation loss: {best_val_loss:.4f}")
    else:
        patience_counter += 1
        print(f"  -> No improvement ({patience_counter}/{early_stopping_patience})")
        
    if patience_counter >= early_stopping_patience:
        print(f"\nEarly stopping triggered after {epoch+1} epochs")
        # Restore best model
        model.load_state_dict(best_model_state)
        break

print(f"\nTraining complete! Best validation loss: {best_val_loss:.4f}")

Epoch 1/350, Train Loss: 1.2066, Val Loss: 7.5373
Epoch 2/350, Train Loss: 0.6921, Val Loss: 7.7605
Epoch 3/350, Train Loss: 0.5242, Val Loss: 8.8215
Epoch 4/350, Train Loss: 0.4467, Val Loss: 9.7248
Epoch 5/350, Train Loss: 0.4041, Val Loss: 8.2627
Epoch 6/350, Train Loss: 0.3733, Val Loss: 6.5564
Epoch 7/350, Train Loss: 0.3252, Val Loss: 8.1244
Epoch 8/350, Train Loss: 0.2925, Val Loss: 9.2705
Epoch 9/350, Train Loss: 0.2936, Val Loss: 7.8831
Epoch 10/350, Train Loss: 0.2553, Val Loss: 9.3561
Epoch 11/350, Train Loss: 0.2396, Val Loss: 8.1766
Epoch 12/350, Train Loss: 0.2474, Val Loss: 8.7651
Epoch 13/350, Train Loss: 0.2144, Val Loss: 8.9936
Epoch 14/350, Train Loss: 0.2059, Val Loss: 8.2401
Epoch 15/350, Train Loss: 0.2019, Val Loss: 9.6034
Epoch 16/350, Train Loss: 0.2011, Val Loss: 9.9862
Epoch 17/350, Train Loss: 0.2015, Val Loss: 8.9193
Epoch 18/350, Train Loss: 0.1873, Val Loss: 9.4462
Epoch 19/350, Train Loss: 0.1857, Val Loss: 11.4432
Epoch 20/350, Train Loss: 0.1885, Val L

### Plot training curves

In [ ]:
# Plot training and validation loss curves
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Training Loss', linewidth=2)
plt.plot(val_losses, label='Validation Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss Over Time')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Calculate final metrics
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

print("\nTest Set Classification Report:")
print("=" * 80)
clasif_rep = classification_report(all_labels, all_preds)
print(clasif_rep)

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       186
           1       0.00      0.00      0.00        12
           2       0.01      0.00      0.01       540
           3       0.00      0.00      0.00        12
           4       0.00      0.00      0.00        58
           5       0.00      0.00      0.00        33
           6       0.00      0.00      0.00        10
           7       0.00      0.00      0.00         4
           8       0.00      0.00      0.00        19
           9       0.00      0.00      0.00        18
          10       0.00      0.00      0.00         2
          11       0.00      0.00      0.00        45
          12       0.00      0.00      0.00         4
          13       0.00      0.00      0.00         8
          14       0.00      0.00      0.00         1
          15       0.00      0.00      0.00         0
          16       0.00      0.00      0.00         0

    accuracy              

/home/Shelby/miniconda3/envs/BlackbirdOP_py312/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/Shelby/miniconda3/envs/BlackbirdOP_py312/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/Shelby/miniconda3/envs/BlackbirdOP_py312/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier,

## Train Random Forest Classifier

Single-label classification: each call embedding maps to one call type.

In [ ]:
# Train a Random Forest classifier for single-label classification
print("Training Random Forest classifier...")
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# Train the model
model.fit(X_train, y_train)
print("Training complete!")

## Evaluate on Validation Set

In [ ]:
# Predict on validation set
y_val_pred = model.predict(X_val)

# Get all unique call types
call_types = sorted(set(y_train) | set(y_val) | set(y_test))

# Calculate per-class metrics
print("Validation Set Performance:")
print("=" * 80)
print(classification_report(y_val, y_val_pred, labels=call_types, zero_division=0))

In [ ]:
# Calculate overall validation accuracy
from sklearn.metrics import accuracy_score

val_accuracy = accuracy_score(y_val, y_val_pred)
print(f"\nOverall Validation Accuracy: {val_accuracy:.4f}")

## Evaluate on Test Set

In [ ]:
# Predict on test set
y_test_pred = model.predict(X_test)

# Calculate per-class metrics
print("Test Set Performance:")
print("=" * 80)
print(classification_report(y_test, y_test_pred, labels=call_types, zero_division=0))

In [ ]:
# Calculate overall test accuracy
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"\nOverall Test Accuracy: {test_accuracy:.4f}")

## Visualize Confusion Matrices

In [ ]:
# Plot confusion matrix for all call types
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_test_pred, labels=call_types)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=call_types, yticklabels=call_types, cmap='Blues')
plt.title('Confusion Matrix - Test Set')
plt.ylabel('True Call Type')
plt.xlabel('Predicted Call Type')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [2]:
# Train Classifier on BirdNET Embeddings

#This notebook trains a shallow classifier on pre-computed BirdNET embeddings using the train/test/validation splits from YOLO.

In [ ]:
# Plot confusion matrices for each call type
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, call_type in enumerate(call_types):
    if i < len(axes):
        cm = confusion_matrix(y_test[:, i], y_test_pred[:, i])
        sns.heatmap(cm, annot=True, fmt='d', ax=axes[i], 
                   xticklabels=['Absent', 'Present'],
                   yticklabels=['Absent', 'Present'],
                   cmap='Blues')
        axes[i].set_title(f'{call_type}')
        axes[i].set_ylabel('True')
        axes[i].set_xlabel('Predicted')

# Hide extra subplots if there are fewer call types than subplots
for i in range(len(call_types), len(axes)):
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## Visualize Confusion Matrices

In [ ]:
# Calculate overall test metrics
print("\nOverall Test Metrics:")
print(f"Accuracy: {accuracy_score(y_test, y_test_pred):.4f}")
print(f"Precision (macro): {precision_score(y_test, y_test_pred, average='macro', zero_division=0):.4f}")
print(f"Recall (macro): {recall_score(y_test, y_test_pred, average='macro', zero_division=0):.4f}")
print(f"F1 Score (macro): {f1_score(y_test, y_test_pred, average='macro', zero_division=0):.4f}")

In [ ]:
# Predict on test set
y_test_pred = model.predict(X_test)

# Calculate per-class metrics
print("Test Set Performance (per call type):")
print("=" * 80)
for i, call_type in enumerate(call_types):
    print(f"\n{call_type}:")
    print(classification_report(y_test[:, i], y_test_pred[:, i], 
                                target_names=['Absent', 'Present'],
                                zero_division=0))

## Evaluate on Test Set

In [ ]:
# Calculate overall accuracy metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("\nOverall Validation Metrics:")
print(f"Accuracy: {accuracy_score(y_val, y_val_pred):.4f}")
print(f"Precision (macro): {precision_score(y_val, y_val_pred, average='macro', zero_division=0):.4f}")
print(f"Recall (macro): {recall_score(y_val, y_val_pred, average='macro', zero_division=0):.4f}")
print(f"F1 Score (macro): {f1_score(y_val, y_val_pred, average='macro', zero_division=0):.4f}")

In [ ]:
# Predict on validation set
y_val_pred = model.predict(X_val)

# Calculate per-class metrics
print("Validation Set Performance (per call type):")
print("=" * 80)
for i, call_type in enumerate(call_types):
    print(f"\n{call_type}:")
    print(classification_report(y_val[:, i], y_val_pred[:, i], 
                                target_names=['Absent', 'Present'],
                                zero_division=0))

## Evaluate on Validation Set

In [ ]:
# Train a Random Forest classifier for multi-label classification
print("Training Random Forest classifier...")
rf_classifier = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# Wrap in MultiOutputClassifier for multi-label
model = MultiOutputClassifier(rf_classifier, n_jobs=-1)

# Train the model
model.fit(X_train, y_train)
print("Training complete!")

## Train Random Forest Classifier

Using a Random Forest wrapped in MultiOutputClassifier for multi-label classification.

In [ ]:
# Check class distribution in each split
call_types = train_labels.columns

print("Class distribution in training set:")
train_dist = pd.DataFrame({
    'Call Type': call_types,
    'Count': y_train.sum(axis=0)
}).sort_values('Count', ascending=False)
print(train_dist)

print("\nClass distribution in validation set:")
val_dist = pd.DataFrame({
    'Call Type': call_types,
    'Count': y_val.sum(axis=0)
}).sort_values('Count', ascending=False)
print(val_dist)

print("\nClass distribution in test set:")
test_dist = pd.DataFrame({
    'Call Type': call_types,
    'Count': y_test.sum(axis=0)
}).sort_values('Count', ascending=False)
print(test_dist)

## Check Class Distribution

In [ ]:
# Find common indices between embeddings and labels
train_common = train_labels.index.intersection(embeddings.index)
val_common = val_labels.index.intersection(embeddings.index)
test_common = test_labels.index.intersection(embeddings.index)

print(f"Training samples with embeddings: {len(train_common)} / {len(train_labels)}")
print(f"Validation samples with embeddings: {len(val_common)} / {len(val_labels)}")
print(f"Test samples with embeddings: {len(test_common)} / {len(test_labels)}")

# Prepare training data
X_train = embeddings.loc[train_common].values
y_train = train_labels.loc[train_common].values

# Prepare validation data
X_val = embeddings.loc[val_common].values
y_val = val_labels.loc[val_common].values

# Prepare test data
X_test = embeddings.loc[test_common].values
y_test = test_labels.loc[test_common].values

print(f"\nX_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}, y_val shape: {y_val.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

## Merge Embeddings with Labels

In [ ]:
# Load train/test/validation splits with labels
train_labels = pd.read_csv('./annotated_data/train_set.csv', index_col=0)
val_labels = pd.read_csv('./annotated_data/val_set.csv', index_col=0)
test_labels = pd.read_csv('./annotated_data/test_set.csv', index_col=0)

print(f"Training labels: {train_labels.shape}")
print(f"Validation labels: {val_labels.shape}")
print(f"Test labels: {test_labels.shape}")
print(f"\nCall types: {list(train_labels.columns)}")

## Load Train/Test/Validation Splits

In [ ]:
# Load BirdNET embeddings
embeddings_path = '/home/Shelby/blackbird_calls/Experiments/Detection_of_calls/Datasets/All_call_classes/Split_data_controlled/OpenSoundScape/birds_embeddings_birdnet_padded_3s.csv'
embeddings = pd.read_csv(embeddings_path, index_col=0)

print(f"Loaded embeddings: {embeddings.shape}")
print(f"First few rows of embeddings index:")
print(embeddings.index[:5])
embeddings.head()

## Load BirdNET Embeddings

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, multilabel_confusion_matrix
from sklearn.multioutput import MultiOutputClassifier
import matplotlib.pyplot as plt
import seaborn as sns